# Notebook 01: Download `DocLayNet` and sub sample it
**Author**: Juan Pablo Triana Martinez
**Date**: 2026-03-12

One of the projects I am working on is to create an **optical character recognition** system. This is divided into the main steps

```Python
pdf_doc -> preprocessed_img -> text_detection -> text_recognition
```
**Before attempting to recognize text**, we need to preprocess the data, as well as obtain boundary boxes where the text is located in order to increase the reliability of the **optical character recogntion system**.

Therefore, this notebook would focus into performing **Text Detection** on the following dataset:

*DocLayNet provides page-by-page layout segmentation ground-truth using bounding-boxes for 11 distinct class labels on 80863 unique pages from 6 document categories.*

- **DocLayNet Datast link**: https://arxiv.org/pdf/2206.01062
- **Github link**: https://github.com/DS4SD/DocLayNet

<img src="https://evermap.com/UserGuides/T1/Tutorial_ABM_CharacterRecognitionInPDF%20(8).PNG" width="800">

### `DocLayNet` Dataset

This dataset contains around 80863 PDFs pages. Across these, we also have 7059 carry two instances of human annotaions, and 1591 carry three. Having a total of 91104 total annotations. We have the following categories, which are (found in **Table 1** from DocLayNet paper):
* Caption ~ 22524
* Footnote ~ 6318
* Formula ~ 25027
* List-item ~ 185660
* Page-footer ~ 70878
* Page-header ~ 58022
* Picture ~ 45976
* Section-header ~ 142884
* Table ~ 34733
* Text ~ 510377
* Title ~ 5071
* **Image**: PIL image of size 1025 y 1025.

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*VokPPkMBfMO-RYHEi7BG6g.png" width="800">

This dataset contains rectangular bounding boxes using COCO format, as shown here: https://cocodataset.org/#format-data

*Without further do, let's prepare the codes in order to download the `DocLayNet` dataset.*

## 1. Create a `download_raw_data` and `extract_raw_data` methods to obtain DocLayNet 

In [1]:
from pathlib import Path
from tqdm import tqdm
import requests

def download_raw_data(
    data_folder_path: Path,
    filename_link:str,
    chunk_size:int = 16) -> Path:
    '''
    The following function will download the raw data from a privided link and save it in the
    specified data_folder_path
    Args:
        data_folder_path (Path): Path where the raw data will be saved.
        filename_link (str): The link to the raw data file.
    '''

    # Create the data_folder_path if it doesn't exist
    data_folder_path.mkdir(parents=True, exist_ok=True)

    # Let's create a raw folder inside the data folder
    raw_folder_path = data_folder_path / "raw"
    raw_folder_path.mkdir(parents=True, exist_ok=True)

    # Download the file and save it in the raw folder
    file_name = filename_link.split("/")[-1] # Select the last split, filename
    file_path = raw_folder_path / file_name

    if file_path.exists():
        print(f"[INFO] File already exists: {file_path}")
        return file_path
    else:
        print(f"[INFO] Downloading raw data from: {filename_link}")

        #Stream Donbwload the file
        with requests.get(filename_link, stream=True) as response:
            response.raise_for_status()

            # Find the total file size and chunk size
            total_size = int(response.headers.get('content-length', 0))
            chunk_size = chunk_size * 1024 * 1024 # 16 MB

            # Open the file and write the chunks to it
            with open(file_path, 'wb') as f, tqdm(
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                desc=file_name
            ) as pbar:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
    print(f"[INFO] Raw data downloaded at: {file_path}")
    return file_path

### 1.1 Get the links for `DocLayNet_core.zip` and `DocLayNet_extra.zip`

Both files contain the following, (taken from the README.md):
1. PNG images of all pages, resized to square 1025 x 1025px
2. Bounding-box annotations in COCO format for each PNG image
3. Extra: Single-page PDF files matching each PNG image
4. Extra: JSON file matching each PDF page, which provides the digital text cells with coordinates and content

In [2]:
# Links for DocLayNet dataset
internet_links = {
    "core": "https://codait-cos-dax.s3.us.cloud-object-storage.appdomain.cloud/dax-doclaynet/1.0.0/DocLayNet_core.zip",
    "extra": "https://codait-cos-dax.s3.us.cloud-object-storage.appdomain.cloud/dax-doclaynet/1.0.0/DocLayNet_extra.zip"
}

In [3]:
# Setup the data folder path
data_folder_path = Path().cwd().parent / "data"

### 1.2 Download `DocLayNet_core.zip`

In [4]:
doclaynet_core_path = download_raw_data(
    data_folder_path=data_folder_path,
    filename_link=internet_links["core"])

[INFO] File already exists: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\data\raw\DocLayNet_core.zip


### 1.3 Download `DocLayNet_extra.zip`

In [5]:
doclaynet_extra_path = download_raw_data(
    data_folder_path=data_folder_path,
    filename_link=internet_links["extra"])

[INFO] File already exists: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\data\raw\DocLayNet_extra.zip


## 2. Let's create the `extract_raw_data` method

Now that we have both our `DocLayNet_core` and `DocLayNet_extra` zip folders, we need to know extract them

In [6]:
import zipfile

def extract_raw_data(
    data_folder_path:Path,
    zip_file_data_path: Path) -> Path:
    '''
    The following function will extract the raw data from a zip file
    into a specified data folder path
    Args:
        data_folder_path (Path): Path where the extracted data will be saved.
        zip_file_data_path (Path): Path where the raw data zip file is located.
    '''

    # Create a folder for extracted data
    extracted_data_path = data_folder_path / "extracted"
    extracted_data_path.mkdir(parents = True, exist_ok=True)

    # Let's obtain the zip local file name
    file_name = zip_file_data_path.stem

    # Let's now extract the data from zipfile
    with zipfile.ZipFile(zip_file_data_path, "r") as zip_ref:
        s = zip_ref.infolist()

        # Iterate over the zip file contents and extract them
        for member in tqdm(s, desc="Extracting files", total = (len(s))):
            output_path = extracted_data_path / file_name / member.filename
            if output_path.exists():
                continue
            zip_ref.extract(member, extracted_data_path/ file_name)
    return extracted_data_path / file_name


### 2.1 Extract `DocLayNet_core`

In [7]:
doclaynet_core_extracted_path = extract_raw_data(
    data_folder_path=data_folder_path,
    zip_file_data_path=doclaynet_core_path)

Extracting files: 100%|██████████| 81478/81478 [00:08<00:00, 9318.56it/s] 


### 2.2 Extract `DocLayNet_extra`

In [8]:
doclaynet_extra_extracted_path = extract_raw_data(
    data_folder_path = data_folder_path,
    zip_file_data_path = doclaynet_extra_path
)

Extracting files: 100%|██████████| 162946/162946 [00:16<00:00, 9874.61it/s] 


## 3. Create a `ObtainSubSample` class
This method is key to create smaller samples from the 80863 possible files:
1. `DocLayNet_core` has a size ~ 30.1 to 30.2 GB!
2. `DocLayNet_extra` has a size ~ 10.8 to 11.1 GB!
3. `DocLayNet_core.zip` has a size ~ 27.9 GB!
4. `DocLayNet_extra.zip` has a size ~ 7.45 GB!

These files are **EXTREMELY** big, which is gonna make **Model Evaluation, and Model Experiments** too slow, shown in the next diagram

<img src="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/01_a_pytorch_workflow.png" width="1000">


Therefore, we are gonna create the `ObtainSubSample` class, but why a class? well, recall that the original DocLayNet Distribution is the following:

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*VokPPkMBfMO-RYHEi7BG6g.png" width="800">

To maintain **a close estimate!, NOTE: This is not taking into account the 11 text categories, rather the split is gonna be based on the document categories**, we are gonna do the following inside the class:
1. Constructor will take a `n_samples_train:int` parameter that sets the total samples retrieved from train dataset, found inside `train.json` of `DocLayNet_core/COCO`.
2. Constructor will take a `n_samples_val:int` parameter that sets the total samples retrieved from train dataset, found inside `val.json` of `DocLayNet_core/COCO`.
3. Constructor will take a `n_samples_test:int` parameter that sets the total samples retrieved from train dataset, found inside `test.json` of `DocLayNet_core/COCO`.
4. Constructor will take a `subsample_name:str` parameter, which will name the folder containing the following.
6. Constructor will take the `local_doclaynet_zip_core_path: Path` parameter, pointing to where the `DocLayNet_core` path is located.
7. Constructor will take the `local_doclaynet_zip_extra_path: Union[Path, None]` parameter, pointing to where the `DocLayNet_extra` path is located. if `added_extra` is False, this would be ignored.
8. Constructor will take a flag named `added_extra:bool = True` parameter, which will tell wether to add the extra files with folders: `JSON` and `PDF`.
9. Class will create a `metadata.txt` file containing all of the previous information, and any additional detail.
10. Class will read from zip file the `.json` train, val, and test from `raw/DocLayNet_core.zip/COCO`.
11. Class will proceed depending on the set `n_samples_train`, `n_samples_val`, and `n_samples_test`, to extract with a specified `seed`, the correspoding `hash` keys.
12. Class will then save these `train_hash_keys`, `val_hash_keys`, and `test_hash_keys`, and stream download ONLY those from `raw/DocLayNet_core.zip/COCO`.

Key thing here is that we will use `assert` statements, to ensure that `n_samples` parameters are lower than their maximum allowed values (i.e: We will find out which are when reading the `.json` files inside `DocLayNet_core/COCO`).

```
subsample_name/
│
├── PNG/ -> Contains all train, val, test
│   ├── <hash>.png
│
├── JSON/ -> Contains all train, val, test
│   ├── <hash>.json
│
├──  coco/
│    └── train.json
│    └── val.json
│    └── test.json
│
└── metadata.txt
```

Without further do, let's begin!

### 3.1 Let's nail the logic of train, val, and test `.json` files
This one is key, in order to retrieve the proper subsample 

In [9]:
# Let's do a small test
import json

# Let's get all of the json files
test_paths = doclaynet_core_extracted_path / "COCO"

assert test_paths.exists(), "DocLayNet_core local path doesn't exist, please add it"

json_files_paths = list(test_paths.rglob("*.json"))

coco_files = []
for json__file in json_files_paths:
    with open(str(json__file)) as f:
        coco = json.load(f)
        coco_files.append(coco)

In [10]:
len(coco_files[2]["images"])

6489

### 3.2 Use of ChatGPT to target -> write
I asked chatgpt to aid me how to target -> write code from the `.zip` files.

In [11]:
"""
CODE GENERATED FROM CHATGPT: 2026-03-13
Guide to stream ZIP extraction
"""

import zipfile
from tqdm import tqdm

zip_path = Path("doclaynet_extra.zip")
extract_dir = Path("data")

with zipfile.ZipFile(zip_path) as zf:

    members = zf.infolist()

    for member in tqdm(members, desc="Streaming extraction"):

        if member.is_dir():
            continue

        file_path = Path(member.filename)
        file_hash = file_path.stem

        if file_hash not in sample_hashes:
            continue

        output_path = extract_dir / file_path
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with zf.open(member) as source, open(output_path, "wb") as target:
            target.write(source.read())


FileNotFoundError: [Errno 2] No such file or directory: 'doclaynet_extra.zip'

In [11]:
from typing import Union, List
import random
import json
import zipfile
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

class ObtainSubSample:
    '''
    This class will perform the following:
        1. Class will read from zip file the `.json` train, val, and test from `raw/DocLayNet_core.zip/COCO`.
        2. Class will proceed depending on the set `n_samples_train`, `n_samples_val`, and `n_samples_test`, to extract with a specified `seed`, the correspoding `hash` keys.
        3. Class will then save these `train_hash_keys`, `val_hash_keys`, and `test_hash_keys`, and stream download ONLY those from `raw/DocLayNet_core.zip/COCO`.
        4. Class will create a `metadata.txt` file containing all of the previous information, and any additional detail. 
    '''
    def __init__(self,
                data_folder_path:Path,
                n_samples_train:int,
                n_samples_val:int,
                n_samples_test:int,
                subsample_name:str,
                local_doclaynet_zip_core_path: Path,
                local_doclaynet_zip_extra_path: Union[Path, None],
                added_extra:bool = True,
                seed:int = 42):
        '''
        Args:
            - data_folder_path (Path): Pathlib Parameter where data folder is located.
            - n_samples_train (int): Parameter that sets the total samples retrieved from train dataset, found inside `train.json` of `DocLayNet_core/COCO`.
            - n_samples_val (int): Parameter that sets the total samples retrieved from train dataset, found inside `val.json` of `DocLayNet_core/COCO`.
            - n_samples_test (int): Parameter that sets the total samples retrieved from train dataset, found inside `test.json` of `DocLayNet_core/COCO`.
            - subsample_name (str): Parameter, which will name the folder containing the following.
            - added_extra (bool) = True parameter, which will tell wether to add the extra files with folders: `JSON` and `PDF`.
            - seed (int): Integer parameter that sets the random sampling from the machine.
        
        Key different functionalities:
            - Constructor will take the `local_doclaynet_zip_core_path: Path` parameter, pointing to where the `DocLayNet_core` path is located.
            - Constructor will take the `local_doclaynet_zip_extra_path: Union[Path, None]` parameter, pointing to where the `DocLayNet_extra` path is located. if `added_extra` is False, this would be ignored.
        '''
        # Let's add some parameters
        self.data_folder_path = data_folder_path
        self.subsample_name = subsample_name + "_seed_" + str(seed)
        self.added_extra = added_extra
        self.seed = seed

        ### ============================== NOTE ========================== ###
        ### This is gonna be used with ChatGPT generated code to integrate moving files
        # Set the paths
        self.local_doclaynet_zip_core_path = local_doclaynet_zip_core_path
        self.local_doclaynet_zip_extra_path = local_doclaynet_zip_extra_path
        ### =============================================================== ###

        # Set the random seed
        self.seed = seed
        random.seed(seed)

        # Set the subsample directories
        self.set_subsample_directories()

        # Get the coco_files
        self.coco_files = self.get_coco_files()

        # Let's obtain the max_number_train_samples
        self.max_num_train_samples = len(self.coco_files["train.json"]["images"])
        self.max_num_val_samples = len(self.coco_files["val.json"]["images"])
        self.max_num_test_samples = len(self.coco_files["test.json"]["images"])

        # Double check that they are set properly
        self.n_samples_train = self.set_num_samples(n_samples_train, self.max_num_train_samples)
        self.n_samples_val = self.set_num_samples(n_samples_val, self.max_num_val_samples)
        self.n_samples_test = self.set_num_samples(n_samples_test, self.max_num_test_samples)

        # Get the coco_key_dictionary containing all json files information
        self.json_coco_dict = self.get_subsample_coco_files_dict()
        self.set_subsample_coco_files()

    def set_subsample_directories(self):
        '''
        Public instance method that sets the directories where all
        the new subsample of data will be located.
        '''
        # Let's create a data folder to
        self.extracted_data_path = self.data_folder_path / self.subsample_name
        self.extracted_data_path.mkdir(parents=True, exist_ok=True)

        # Let's create the directories
        self.coco_path = self.extracted_data_path / "COCO"
        self.coco_path.mkdir(parents=True, exist_ok=True)
        self.png_path = self.extracted_data_path / "PNG"
        self.png_path.mkdir(parents=True, exist_ok=True)
        if self.added_extra:
            self.json_extra_path = self.extracted_data_path / "JSON"
            self.json_extra_path.mkdir(parents=True, exist_ok=True)
            self.pdf_extra_path = self.extracted_data_path / "PDF"
            self.pdf_extra_path.mkdir(parents=True, exist_ok=True)
    
    def get_coco_files(self) -> dict[str, dict[str, str]]:
        '''
        Public instance method that gets the .json train, val, and test
        json files and retrieves dictionaries
        '''
        # Let's now extract the json files from the .zip core
        with zipfile.ZipFile(self.local_doclaynet_zip_core_path, "r") as zip_ref:
            s = zip_ref.infolist()

            # Iterate over the zip file contents and extract them
            coco_files = {}
            for member in s:
                if ".json" in member.filename:
                    print(f"[INFO] Extracting {member.filename.split("/")[-1]} for {self.subsample_name}...")
                    with zip_ref.open(member) as f:
                        coco = json.load(f)
                        coco_files[member.filename.split("/")[-1]] = coco
                    print(f"[INFO] Extracted {member.filename.split("/")[-1]} for {self.subsample_name}")
                else:
                    continue
        return coco_files

    def set_num_samples(self, n_samples_passed:int, max_num_passed:int) -> int:
        '''
        Public Instance method that, depending on the comparison 
        between n_samples_passed and max_num_passed, perform:
        - if n_samples_passed, return an error that it must be greater than 0
        - if n_samples_passed <= max_num_passed, n_samples passed stays the same
        - otherwise n_samples_passed is set to to max_num_passed
        '''
        assert n_samples_passed > 0, f"Please add a valid positive value of n_samples_passed"
        if n_samples_passed <= max_num_passed:
            return n_samples_passed
        else:
            return min(n_samples_passed, max_num_passed)
    
    def get_subsample_coco_files_dict(self) -> dict[str, dict[str, List[dict]]]:
        '''
        Public instanec method that returns the json files
        in a dictionary format to be uploaded to the new 
        ''' 

        test_coco_holder = {}

        # Let's iterate acorss each key
        for key in self.coco_files.keys():
            # We are creating a new dictionary notebook 
            json_coco_dict = {}

            # Let's add the categories as we know
            print(f"[INFO] Getting categories to: {key}...")
            json_coco_dict["categories"] = self.coco_files[key]["categories"]

            # Let's now sample according to if its train, val, or test
            if "train" in key:
                print(f"[INFO] Getting subsampled images, size: {self.n_samples_train}, to: {key}...")
                images_sampled = random.sample(self.coco_files[key]["images"], self.n_samples_train)
            elif "val" in key:
                print(f"[INFO] Getting subsampled images, size: {self.n_samples_val}, to: {key}...")
                images_sampled = random.sample(self.coco_files[key]["images"], self.n_samples_val)
            elif "test" in key:
                print(f"[INFO] Getting subsampled images, size: {self.n_samples_test}, to: {key}...")
                images_sampled = random.sample(self.coco_files[key]["images"], self.n_samples_test)
            else:
                print(f"[INFO] Getting entire image set, size: {len(self.coco_files[key]["images"])}, to: {key}...")
                images_sampled = self.coco_files[key]["images"]
            
            # Assign them properly
            json_coco_dict["images"] = images_sampled
            print(f"[INFO] Obtained subsampled complete!")

            # Obtain selected image ids inside a set
            print(f"[INFO] Getting annotations from subsample...")
            sel_images_sampled_ids = set([x["id"] for x in images_sampled])

            # Retrieve the annotations_sampled from sel_images_sampled_ids
            annotations_sampled = [x for x in self.coco_files[key]["annotations"] if x["image_id"] in sel_images_sampled_ids]

            # Add them properly
            json_coco_dict["annotations"] = annotations_sampled
            print(f"[INFO] Obtained annotations!")

            # Append them to test_coco_holder
            test_coco_holder[key] = json_coco_dict
        
        return test_coco_holder

    def set_subsample_coco_files(self):
        '''
        Public instance method that will now write the new .json files inside the folder
        '''
        for coco_filename in self.coco_files.keys():
            # Output path for new filename
            output_path = self.coco_path / coco_filename

            # Data selection
            sel_data = self.json_coco_dict[coco_filename]

            # Now data upload
            with open(output_path, "w") as f:
                print(f"[INFO] Adding subsample .json file, {coco_filename}..")
                json.dump(sel_data, f, indent = 4)
                print(f"[INFO] Added subsample .json file, {coco_filename}!")
    
    def get_metadata_file(self):
        '''
        Public instance method that will now write the metadata.txt file
        '''

    def extract_core_files(self):
        '''
        Public instance method that will now extract the following:
        1. from DocLayNet_core.zip path, extract the PNG files
        '''

        # Let's get the hash keys that are from the subsample
        for key in self.json_coco_dict:
            
            # Let's get the data from specific key (i.e: train.json, val.json, test.json)
            data_sel = self.json_coco_dict[key]

            # Let's get the sample hashes
            subsample_hashes = set([x["file_name"].replace(".png", "") for x in data_sel["images"]])

            # Let's do first the Doclaynet_core.zip to get the .png files
            print(f"[INFO]: Extracting core files...")
            with zipfile.ZipFile(self.local_doclaynet_zip_core_path) as zip_core:
                s = zip_core.infolist()

                for member in tqdm(s, desc=f"Extracting subsample", total = len(s)):
                    if "__MACOSX" in member.filename:
                        continue
                    if member.is_dir():
                        continue
                    if ".png" in member.filename:             
                        # Get the filename inside zip without path
                        file_name = member.filename.split("/")[-1].replace(".png", "")

                        # if file_name is inside the subsample_hashes, write -> target from DoclayNet_core.zip
                        if file_name in subsample_hashes:
                            download_filename = file_name + ".png"
                            output_path = self.png_path / download_filename

                            # If previously downloaded, continue
                            if output_path.exists():
                                continue
                            with zip_core.open(member) as source, open(output_path, "wb") as target:
                                target.write(source.read())
            print(f"[INFO]: Core files extracted!")

    def extract_extra_files(self):
        '''
        Public instance method that will now extract the following:
        1. from DocLayNet_core.zip path, extract the PNG files
        '''

        if self.added_extra:
            # Let's get the hash keys that are from the subsample
            for key in self.json_coco_dict:
                
                # Let's get the data from specific key (i.e: train.json, val.json, test.json)
                data_sel = self.json_coco_dict[key]

                # Let's get the sample hashes
                subsample_hashes = set([x["file_name"].replace(".png", "") for x in data_sel["images"]])

                # Let's do first the Doclaynet_extra.zip to get the .png files
                print(f"[INFO]: Extracting extra files...")
                with zipfile.ZipFile(self.local_doclaynet_zip_extra_path) as zip_core:
                    s = zip_core.infolist()

                    for member in tqdm(s, desc=f"Extracting subsample", total = len(s)):
                        if "__MACOSX" in member.filename:
                            continue

                        if member.is_dir():
                            continue

                        if ".json" in member.filename:             
                            # Get the filename inside zip without path
                            file_name = member.filename.split("/")[-1].replace(".json", "")

                            # if file_name is inside the subsample_hashes, write -> target from DoclayNet_core.zip
                            if file_name in subsample_hashes:
                                download_filename = file_name + ".json"
                                output_path = self.json_extra_path / download_filename

                                # If previously downloaded, continue
                                if output_path.exists():
                                    continue
                                with zip_core.open(member) as source, open(output_path, "wb") as target:
                                    target.write(source.read())

                        if ".pdf" in member.filename:             
                            # Get the filename inside zip without path
                            file_name = member.filename.split("/")[-1].replace(".pdf", "")

                            # if file_name is inside the subsample_hashes, write -> target from DoclayNet_core.zip
                            if file_name in subsample_hashes:
                                download_filename = file_name + ".pdf"
                                output_path = self.pdf_extra_path / download_filename

                                # If previously downloaded, continue
                                if output_path.exists():
                                    continue
                                with zip_core.open(member) as source, open(output_path, "wb") as target:
                                    target.write(source.read())
                print(f"[INFO]: extra files extracted!")
        else:
            print(f"[INFO]: Extra files is set to False -> {self.added_extra}")

    def write_metadata(self):
        '''
        Public instance method that writes a metadata.txt file
        containing information about the subsample creation.
        '''

        metadata_path = self.extracted_data_path / "metadata.txt"

        # Count extracted files
        num_png = len(list(self.png_path.glob("*.png")))

        num_json_extra = 0
        num_pdf_extra = 0

        if self.added_extra:
            num_json_extra = len(list(self.json_extra_path.glob("*.json")))
            num_pdf_extra = len(list(self.pdf_extra_path.glob("*.pdf")))

        # Metadata text
        metadata_text = f"""
        ==============================
        DocLayNet Subsample Metadata
        ==============================

        Subsample Name:
        {self.subsample_name}

        Creation Timestamp:
        {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

        Random Seed:
        {self.seed}

        ------------------------------
        Dataset Sizes
        ------------------------------

        Max Train Samples: {self.max_num_train_samples}
        Max Val Samples: {self.max_num_val_samples}
        Max Test Samples: {self.max_num_test_samples}

        Selected Train Samples: {self.n_samples_train}
        Selected Val Samples: {self.n_samples_val}
        Selected Test Samples: {self.n_samples_test}

        ------------------------------
        Files Extracted
        ------------------------------

        PNG Images Extracted: {num_png}

        Extra Files Enabled: {self.added_extra}

        JSON Extra Files Extracted: {num_json_extra}
        PDF Extra Files Extracted: {num_pdf_extra}

        ------------------------------
        Dataset Paths
        ------------------------------

        Core Dataset Zip:
        {self.local_doclaynet_zip_core_path}

        Extra Dataset Zip:
        {self.local_doclaynet_zip_extra_path}

        Output Dataset Directory:
        {self.extracted_data_path}

        ------------------------------
        Directory Structure
        ------------------------------

        COCO Annotations: {self.coco_path}
        PNG Images: {self.png_path}
        """

        if self.added_extra:
            metadata_text += f"""
            Extra JSON Files: {self.json_extra_path}
            Extra PDF Files: {self.pdf_extra_path}
            """

        # Write file
        with open(metadata_path, "w") as f:
            print("[INFO] Writing metadata file...")
            f.write(metadata_text.strip())

        print(f"[INFO] Metadata file written to {metadata_path}")


In [12]:
sub_sampler = ObtainSubSample(
    data_folder_path = data_folder_path,
    n_samples_train = 20,
    n_samples_val = 3,
    n_samples_test = 3,
    subsample_name = "test_subsample",
    local_doclaynet_zip_core_path = doclaynet_core_path,
    local_doclaynet_zip_extra_path = doclaynet_extra_path,
    added_extra = True,
    seed=42)

[INFO] Extracting test.json for test_subsample_seed_42...
[INFO] Extracted test.json for test_subsample_seed_42
[INFO] Extracting train.json for test_subsample_seed_42...
[INFO] Extracted train.json for test_subsample_seed_42
[INFO] Extracting val.json for test_subsample_seed_42...
[INFO] Extracted val.json for test_subsample_seed_42
[INFO] Getting categories to: test.json...
[INFO] Getting subsampled images, size: 3, to: test.json...
[INFO] Obtained subsampled complete!
[INFO] Getting annotations from subsample...
[INFO] Obtained annotations!
[INFO] Getting categories to: train.json...
[INFO] Getting subsampled images, size: 20, to: train.json...
[INFO] Obtained subsampled complete!
[INFO] Getting annotations from subsample...
[INFO] Obtained annotations!
[INFO] Getting categories to: val.json...
[INFO] Getting subsampled images, size: 3, to: val.json...
[INFO] Obtained subsampled complete!
[INFO] Getting annotations from subsample...
[INFO] Obtained annotations!
[INFO] Adding subsamp

### 3.2. Use the `extract_core_files` method to get the `.png` files from `DocLayNet_core.zip`

In [13]:
sub_sampler.extract_core_files()

[INFO]: Extracting core files...


Extracting subsample: 100%|██████████| 81478/81478 [00:00<00:00, 1537667.39it/s]


[INFO]: Core files extracted!
[INFO]: Extracting core files...


Extracting subsample: 100%|██████████| 81478/81478 [00:00<00:00, 995234.15it/s]


[INFO]: Core files extracted!
[INFO]: Extracting core files...


Extracting subsample: 100%|██████████| 81478/81478 [00:00<00:00, 1392138.30it/s]

[INFO]: Core files extracted!


### 3.3. Use the `extract_extra_files` method to get the `.pdf` and `json` files from `DocLayNet_extra.zip`

In [14]:
sub_sampler.extract_extra_files()

[INFO]: Extracting extra files...


Extracting subsample: 100%|██████████| 162946/162946 [00:00<00:00, 1258906.18it/s]


[INFO]: extra files extracted!
[INFO]: Extracting extra files...


Extracting subsample: 100%|██████████| 162946/162946 [00:00<00:00, 1177669.28it/s]


[INFO]: extra files extracted!
[INFO]: Extracting extra files...


Extracting subsample: 100%|██████████| 162946/162946 [00:00<00:00, 1198820.31it/s]


[INFO]: extra files extracted!


### 3.4 Use the `write_metadata` function to get the `metadata.txt` file

In [15]:
sub_sampler.write_metadata()

[INFO] Writing metadata file...
[INFO] Metadata file written to c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\data\test_subsample_seed_42\metadata.txt
